In [1]:
from test.helpers import SDE, simple_batch_sde_solve

import diffrax
import jax
import jax.numpy as jnp
import jax.random as jr
import numpy as np
from diffrax import (
    ControlTerm,
    MultiTerm,
    ODETerm,
    ShARK,
    SpaceTimeLevyArea,
)
from jax import Array


jax.config.update("jax_enable_x64", True)


# Save w, hh and sst to a file
def save_sst(w, hh, sst: Array):
    w_float = float(w)
    hh_float = float(hh)
    w = np.array(w)
    hh = np.array(hh)
    sst = np.array(sst)
    file_name = f"sst_saved_values/sst_w{w_float:.3}_hh{hh_float:.3}.npy"
    with open(file_name, "wb") as f:
        np.save(f, w)
        np.save(f, hh)
        np.save(f, sst)


def load_sst(w_float: float, hh_float: float):
    file_name = f"sst_saved_values/sst_w{w_float:.3}_hh{hh_float:.3}.npy"
    with open(file_name, "rb") as f:
        w = np.load(f)
        hh = np.load(f)
        sst = np.load(f)

    return w, hh, sst


# Define the SSD for computing C_t := \int_0^t W_s ds
# sst_sde is two-dimensional, the first dimension being W_t, the second \int_0^t W_s ds
def drift(t, y, args):
    return jnp.array([0.0, y[0] ** 2], dtype=y.dtype)


def diffusion(t, y, args):
    return jnp.array([1.0, 0.0], dtype=y.dtype)


def get_terms(bm):
    return MultiTerm(ODETerm(drift), ControlTerm(diffusion, bm))


y0 = jnp.array([0.0, 0.0], dtype=jnp.float64)
args = None
t0 = 0.0
t1 = 1.0
bm_shape = ()

sst_sde = SDE(get_terms, args, y0, t0, t1, bm_shape)


def compute_sst(w, hh, num_samples, key):
    keys = jr.split(key, num_samples)
    saveat = diffrax.SaveAt(t1=True)
    bm_tol = 2**-6
    dt0 = 2**-4
    solver = ShARK()
    controller = None
    levy_area = SpaceTimeLevyArea
    if w is not None:
        w = jnp.array(w, dtype=jnp.float64)
    if hh is not None:
        hh = jnp.array(hh, dtype=jnp.float64)
    y1, _ = simple_batch_sde_solve(
        keys,
        sst_sde,
        solver,
        levy_area,
        dt0,
        controller,
        bm_tol,
        saveat,
        _w_t1=w,
        _hh_t1=hh,
    )
    sst = jnp.squeeze(y1)[:, 1]
    return sst


def true_cond_stats_c(w, hh):
    w2 = w**2
    hh2 = hh**2
    mean = 1 / 3 * w2 + w * hh + 6 / 5 * hh2 + 1 / 15
    var = 11 / 6300 + 1 / 180 * w2 + 1 / 175 * hh2
    return mean, var


def stat_error(c: Array, w, hh):
    w = float(w)
    hh = float(hh)
    true_mean, true_var = true_cond_stats_c(w, hh)
    mean_error = jnp.abs(true_mean - jnp.mean(c))
    var_error = jnp.abs(true_var - jnp.var(c))
    return mean_error, var_error

In [2]:
# First unconditioned samples

key = jr.key(775)
sst_uncond = np.array(compute_sst(None, None, 2**22, key))
file_name = "sst_unconditional.npy"
with open(file_name, "wb") as f:
    np.save(f, sst_uncond)

In [15]:
import math


key = jr.key(777)
keys = jr.split(key, 10)
for i in range(10):
    key_w, key_hh = jr.split(keys[i], 2)
    bm_mult = 1.0
    hh_mult = bm_mult * math.sqrt(1 / 12)
    w = bm_mult * jr.normal(key_w, (), dtype=jnp.float64)
    hh = hh_mult * jr.normal(key_hh, (), dtype=jnp.float64)
    sst = compute_sst(w, hh, 2**20, jr.PRNGKey(0))
    mean_error, var_error = stat_error(sst, w, hh)
    print(f"Mean error: {mean_error}, variance error: {var_error}")
    save_sst(w, hh, sst)

Mean error: 0.00036159472519291247, variance error: 7.935701113718227e-05
Mean error: 0.00035374238948254755, variance error: 7.925202993634521e-05
Mean error: 0.0006158253005309478, variance error: 8.283692705801302e-05
Mean error: 0.0004818153193736907, variance error: 8.300312541983327e-05
Mean error: 0.00017089608354425856, variance error: 9.00609599249879e-05
Mean error: 0.00025113669348419965, variance error: 9.288250390053964e-05
Mean error: 5.179031604973794e-05, variance error: 0.00010130854228495093
Mean error: 0.0001865954574095524, variance error: 7.904683324458069e-05
Mean error: 0.0005249324981546066, variance error: 8.166005590211963e-05
Mean error: 0.00026534091358360046, variance error: 8.982529028052182e-05
